# Chapter 6: XGBoost Hyperparameters

**Dataset**: https://archive.ics.uci.edu/ml/datasets/Heart+Disease

In [4]:
# Import the pandas library and give it a shorter alias "pd" for convenience
import pandas as pd

# Use pandas to read a CSV (comma-separated values) file named 'heart_disease.csv'
# This loads the data into a DataFrame, which is like a table with rows and columns
df = pd.read_csv('heart_disease.csv')
# Alternatively ..although set looks different
# df = pd.read_csv('http://bit.ly/kaggletrain')

# Show the first 5 rows of the DataFrame to get a quick look at the data
# Useful to understand what kind of data and columns are present
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


In [5]:
# Display a summary of the DataFrame, including:
# - Number of rows and columns
# - Column names
# - Data types of each column (e.g., int64, float64, object)
# - Number of non-null (non-missing) entries in each column
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    int64  
 1   sex       303 non-null    int64  
 2   cp        303 non-null    int64  
 3   trestbps  303 non-null    int64  
 4   chol      303 non-null    int64  
 5   fbs       303 non-null    int64  
 6   restecg   303 non-null    int64  
 7   thalach   303 non-null    int64  
 8   exang     303 non-null    int64  
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    int64  
 11  ca        303 non-null    int64  
 12  thal      303 non-null    int64  
 13  target    303 non-null    int64  
dtypes: float64(1), int64(13)
memory usage: 33.3 KB


**NOTE:** All data points are non-null and numerical. Hence data is machine learning-ready.

In [6]:
# Import the XGBClassifier class from the xgboost library
# This is a powerful machine learning model used for classification tasks (predicting categories)
from xgboost import XGBClassifier

# Import the accuracy_score function from scikit-learn
# This function helps us measure how many predictions were correct out of all predictions made
from sklearn.metrics import accuracy_score

Declare X as the **predictor** columns and y as the **target** column

In [7]:
# Split the DataFrame into input features (X) and the target label (y)

# X contains all columns **except the last one** — these are the input features used to make predictions
X = df.iloc[:, :-1]

# y contains **only the last column** — this is the target or label we want to predict (e.g., presence of heart disease)
y = df.iloc[:, -1]

**Features (X)**: What we use to predict (like age, cholesterol, blood pressure).

**Target (y)**: What we want to predict (e.g., whether someone has heart disease).

In [8]:
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

X shape: (303, 13)
y shape: (303,)


In [9]:
# Create an XGBoost classification model using decision trees (gbtree)

model = XGBClassifier(
    booster='gbtree',              # Use gradient-boosted decision trees as the underlying model (most common choice)
    objective='binary:logistic',   # This means it's a binary classification task (e.g., 0 = no disease, 1 = disease)
    random_state=2                 # Set a random seed for reproducibility — ensures the same results every time you run
)


---

### 🧠 ***Summary:***

* `XGBClassifier(...)` sets up the model.
* `booster='gbtree'` tells XGBoost to use decision trees.
* `objective='binary:logistic'` means the output will be a probability between 0 and 1, suitable for **binary classification**.
* `random_state=2` is used to control randomness — helpful for debugging and comparing results.

---

## **Get the baseline score**

In [10]:
# Import the cross_val_score function from scikit-learn
# This helps us evaluate the model using cross-validation (splitting the data into parts to test stability)
from sklearn.model_selection import cross_val_score

# Import the numpy library and give it a shorter name "np"
# We'll use it to round numbers and do numeric calculations
import numpy as np

# Perform 5-fold cross-validation using our model and data
# - The data is split into 5 parts
# - The model trains on 4 parts and tests on the 1 remaining part
# - This is repeated 5 times so each part is used as the test once
# - Returns an array of 5 accuracy scores
scores = cross_val_score(model, X, y, cv=5)

# Print all 5 accuracy scores (one for each fold), rounded to 2 decimal places
print('Baseline Accuracy:', np.round(scores, 2))

# Print the average accuracy across the 5 folds
print('Baseline Accuracy mean: %0.2f' % (scores.mean()))

# Accuracy: [0.84 0.85 0.82 0.8  0.77]
# Accuracy mean: 0.81

Baseline Accuracy: [0.82 0.8  0.77 0.78 0.77]
Baseline Accuracy mean: 0.79


***When compared to DecisionTreeClassifier***

An accuracy score of 81% is an excellent starting point, considerably higher than the 76% cross-validation obtained by **DecisionTreeClassifier**

## **Using StratifiedKFold**

This is used to keep class proportions balanced in each fold

see: https://courses.dataschool.io/view/courses/scikit-learn-tips/986254-model-evaluation/2709355-31-shuffle-your-dataset-when-using-cross_val_score

In [11]:
# Import StratifiedKFold from scikit-learn
# This is a smarter version of K-Fold cross-validation that keeps class proportions balanced in each fold
from sklearn.model_selection import StratifiedKFold

# Create a StratifiedKFold cross-validator object
kfold = StratifiedKFold(
    n_splits=5,         # Split the dataset into 5 parts (folds)
    shuffle=True,       # Shuffle the data before splitting (helps avoid bias from ordered data)
    random_state=2      # Fix the shuffling so the results are reproducible (same every run)
)

In [12]:
# Perform cross-validation using the model, input features (X), and target labels (y)
# Use the StratifiedKFold object (kfold) to ensure balanced class splits in each fold
# This returns an array of 5 accuracy scores — one for each fold
scores = cross_val_score(model, X, y, cv=kfold)

# Print the 5 individual accuracy scores, rounded to 2 decimal places
# Helps you see how the model performs across different parts of the data
print('Accuracy:', np.round(scores, 2))

# Print the average accuracy across all 5 folds
# This gives a more stable and general estimate of the model's performance
print('Accuracy mean: %0.2f' % (scores.mean()))

# Accuracy: [0.72 0.82 0.75 0.8  0.82]
# Accuracy mean: 0.78

Accuracy: [0.72 0.77 0.75 0.85 0.87]
Accuracy mean: 0.79


---

### 🐭 ***Summary:***

* `cross_val_score(...)` evaluates your model’s performance more reliably by testing it on different parts of the data.
* `StratifiedKFold` ensures each split has a balanced representation of each class.
* The final **mean accuracy** is a key metric to report in binary classification problems like heart disease prediction.

---

## **Using GridSearchCV**

In [13]:
# Import GridSearchCV and RandomizedSearchCV from scikit-learn
# These are tools to automatically find the best hyperparameters for your model
# GridSearchCV tries every combination from a specified set of values (exhaustive search)
# RandomizedSearchCV tries a fixed number of random combinations (faster for large search spaces)
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold

# Create a StratifiedKFold object for use in model evaluation during hyperparameter tuning
# StratifiedKFold ensures that each fold in cross-validation has a similar distribution of class labels (e.g., 0s and 1s)
kfold = StratifiedKFold(
    n_splits=5,         # Split the dataset into 5 parts (folds)
    shuffle=True,       # Shuffle the data before splitting (important to reduce bias)
    random_state=2      # Set seed for reproducibility — same splits every time you run the code
)


---

### 🧠 ***Summary:***

* `GridSearchCV` and `RandomizedSearchCV` are tools to **optimize your model's hyperparameters**.
* `StratifiedKFold` helps make sure cross-validation results are **balanced and reproducible**, especially in classification tasks.

---

In [14]:
def grid_search(params, random=False):
    """
    Perform hyperparameter tuning for an XGBoost classifier using GridSearchCV or RandomizedSearchCV.

    Parameters:
    - params: dictionary of hyperparameters to search over
    - random: if True, use RandomizedSearchCV; otherwise, use GridSearchCV
    """

    # Create an XGBoost classifier for binary classification
    # Set random_state for reproducibility
    xgb = XGBClassifier(booster='gbtree', objective='binary:logistic', random_state=2)

    # Set up stratified 5-fold cross-validation
    # Ensures class balance in each fold and reproducible splits
    kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=2)

    # Use RandomizedSearchCV if random=True, else use GridSearchCV
    if random:
        # Randomly sample 20 combinations of hyperparameters
        # n_jobs=-1 uses all available CPU cores for faster computation
        grid = RandomizedSearchCV(
            xgb, params, cv=kfold, n_iter=20, n_jobs=-1, random_state=2
        )
    else:
        # Try every combination of hyperparameters (exhaustive search)
        grid = GridSearchCV(
            xgb, params, cv=kfold, n_jobs=-1
        )

    # Fit the search object on the entire dataset (X and y)
    grid.fit(X, y)

    # Extract the best hyperparameter combination found
    best_params = grid.best_params_

    # Print the best parameters
    print("Best params:", best_params)

    # Get the best cross-validation score (mean accuracy across folds)
    best_score = grid.best_score_

    # Print the best score, formatted to 5 decimal places
    print("Best score: {:.5f}".format(best_score))

#### ***GridSearchCV vs RandomizedSearchCV***
**GridSearchCV** exhaustively tests all parameter combinations, while **RandomizedSearchCV** samples a fixed number of random combinations for faster tuning.

## **Tuning with hyperparameters**

### ***1. n_estimators***

In [15]:
# Call the grid_search function with a set of values for the 'n_estimators' hyperparameter
# 'n_estimators' defines the number of boosting rounds (i.e., how many trees to build)

grid_search(params={
    'n_estimators': [100, 200, 400, 800]  # Try each of these values to find the best one
})

# Best params: {'n_estimators': 100}
# Best score: 0.78235

Best params: {'n_estimators': 400}
Best score: 0.79574


---

#### 🧠 ***Summary***:

* This will perform **grid search** using `GridSearchCV` (since `random=False` by default).
* It will test the model with **4 different values** of `n_estimators`: 100, 200, 400, and 800.
* For each value, it runs **5-fold cross-validation** (as defined by `StratifiedKFold`).
* Finally, it prints:

  * The `n_estimators` value that gave the **best average accuracy**
  * The corresponding **accuracy score**

---

### ***2. learning_rate***

In [16]:
# Run grid search to find the best value for the 'learning_rate' hyperparameter
# 'learning_rate' controls how much each tree contributes to the final prediction
# Smaller values make learning slower but may improve generalization
# Larger values speed up learning but may risk overfitting

grid_search(params={
    'learning_rate': [0.01, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]
})

# Best params: {'learning_rate': 0.05}
# Best score: 0.79585

Best params: {'learning_rate': 0.05}
Best score: 0.80235


---

### 🧠 ***Summary:***

* This runs a **GridSearchCV** over **7 values of `learning_rate`**.
* For each value, the model is trained and validated using **Stratified 5-fold CV**.
* The function prints:

  * The best `learning_rate` that produced the highest average accuracy.
  * The corresponding accuracy score.

### 💡 ***Quick Tip:***

* Common practice: start with a smaller learning rate like `0.1` or `0.05` and pair it with a larger number of `n_estimators`.

### 🤺 ***Try Out:***

* Plan to tune both `n_estimators` and `learning_rate` together

---

In [ ]:
grid_search(params={'max_depth':[2, 3, 5, 6, 8]})

Best params: {'max_depth': 2}
Best score: 0.79902


In [ ]:
grid_search(params={'gamma':[0, 0.01, 0.1, 0.5, 1, 2]})

Best params: {'gamma': 0.5}
Best score: 0.79574


In [ ]:
grid_search(params={'min_child_weight':[1, 2, 3, 4, 5]})

Best params: {'min_child_weight': 5}
Best score: 0.81219


In [ ]:
grid_search(params={'subsample':[0.5, 0.7, 0.8, 0.9, 1]})

Best params: {'subsample': 0.8}
Best score: 0.79579


In [ ]:
grid_search(params={'colsample_bytree':[0.5, 0.7, 0.8, 0.9, 1]})

Best params: {'colsample_bytree': 0.7}
Best score: 0.79902


In [ ]:
# Import train_test_split
from sklearn.model_selection import train_test_split

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=2)

In [ ]:
model = XGBClassifier(booster='gbtree', objective='binary:logistic', random_state=2)
eval_set = [(X_test, y_test)]
eval_metric='error'
model.fit(X_train, y_train, eval_metric=eval_metric, eval_set=eval_set)
# make predictions for test data
y_pred = model.predict(X_test)
# evaluate predictions
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy: %.2f%%" % (accuracy * 100.0))

[0]	validation_0-error:0.15790
[1]	validation_0-error:0.10526
[2]	validation_0-error:0.11842
[3]	validation_0-error:0.13158
[4]	validation_0-error:0.11842
[5]	validation_0-error:0.14474
[6]	validation_0-error:0.14474
[7]	validation_0-error:0.14474
[8]	validation_0-error:0.14474
[9]	validation_0-error:0.14474
[10]	validation_0-error:0.14474
[11]	validation_0-error:0.15790
[12]	validation_0-error:0.15790
[13]	validation_0-error:0.17105
[14]	validation_0-error:0.17105
[15]	validation_0-error:0.17105
[16]	validation_0-error:0.15790
[17]	validation_0-error:0.17105
[18]	validation_0-error:0.15790
[19]	validation_0-error:0.17105
[20]	validation_0-error:0.17105
[21]	validation_0-error:0.17105
[22]	validation_0-error:0.18421
[23]	validation_0-error:0.18421
[24]	validation_0-error:0.17105
[25]	validation_0-error:0.18421
[26]	validation_0-error:0.18421
[27]	validation_0-error:0.18421
[28]	validation_0-error:0.18421
[29]	validation_0-error:0.18421
[30]	validation_0-error:0.18421
[31]	validation_0-

In [ ]:
model = XGBClassifier(booster='gbtree', objective='binary:logistic', random_state=2)
eval_set = [(X_test, y_test)]
eval_metric="error"
model.fit(X_train, y_train, eval_metric=eval_metric, eval_set=eval_set, early_stopping_rounds=10, verbose=True)
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy: %.2f%%" % (accuracy * 100.0))

[0]	validation_0-error:0.15790
Will train until validation_0-error hasn't improved in 10 rounds.
[1]	validation_0-error:0.10526
[2]	validation_0-error:0.11842
[3]	validation_0-error:0.13158
[4]	validation_0-error:0.11842
[5]	validation_0-error:0.14474
[6]	validation_0-error:0.14474
[7]	validation_0-error:0.14474
[8]	validation_0-error:0.14474
[9]	validation_0-error:0.14474
[10]	validation_0-error:0.14474
[11]	validation_0-error:0.15790
Stopping. Best iteration:
[1]	validation_0-error:0.10526

Accuracy: 89.47%


In [ ]:
model = XGBClassifier(random_state=2, n_estimators=5000)
eval_set = [(X_test, y_test)]
eval_metric="error"
model.fit(X_train, y_train, eval_metric=eval_metric, eval_set=eval_set, early_stopping_rounds=100)
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy: %.2f%%" % (accuracy * 100.0))

[0]	validation_0-error:0.15790
Will train until validation_0-error hasn't improved in 100 rounds.
[1]	validation_0-error:0.10526
[2]	validation_0-error:0.11842
[3]	validation_0-error:0.13158
[4]	validation_0-error:0.11842
[5]	validation_0-error:0.14474
[6]	validation_0-error:0.14474
[7]	validation_0-error:0.14474
[8]	validation_0-error:0.14474
[9]	validation_0-error:0.14474
[10]	validation_0-error:0.14474
[11]	validation_0-error:0.15790
[12]	validation_0-error:0.15790
[13]	validation_0-error:0.17105
[14]	validation_0-error:0.17105
[15]	validation_0-error:0.17105
[16]	validation_0-error:0.15790
[17]	validation_0-error:0.17105
[18]	validation_0-error:0.15790
[19]	validation_0-error:0.17105
[20]	validation_0-error:0.17105
[21]	validation_0-error:0.17105
[22]	validation_0-error:0.18421
[23]	validation_0-error:0.18421
[24]	validation_0-error:0.17105
[25]	validation_0-error:0.18421
[26]	validation_0-error:0.18421
[27]	validation_0-error:0.18421
[28]	validation_0-error:0.18421
[29]	validation

In [ ]:
grid_search(params={'n_estimators':[2, 25, 50, 75, 100]})

Best params: {'n_estimators': 50}
Best score: 0.78907


In [ ]:
grid_search(params={'max_depth':[1, 2, 3, 4, 6, 7, 8],
                    'n_estimators':[50]})

Best params: {'max_depth': 1, 'n_estimators': 50}
Best score: 0.83869


In [ ]:
grid_search(params={'max_depth':[1, 2, 3, 4, 6, 7, 8],
                    'n_estimators':[2, 50, 100]})

Best params: {'max_depth': 1, 'n_estimators': 50}
Best score: 0.83869


In [ ]:
grid_search(params={'learning_rate':[0.01, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5],
                    'max_depth':[1],
                    'n_estimators':[50]})

Best params: {'learning_rate': 0.3, 'max_depth': 1, 'n_estimators': 50}
Best score: 0.83869


In [ ]:
grid_search(params={'min_child_weight':[1, 2, 3, 4, 5],
                    'max_depth':[1],
                    'n_estimators':[50]})

Best params: {'max_depth': 1, 'min_child_weight': 1, 'n_estimators': 50}
Best score: 0.83869


In [ ]:
grid_search(params={'subsample':[0.5, 0.6, 0.7, 0.8, 0.9, 1],
                    'max_depth':[1],
                    'n_estimators':[50]})

Best params: {'max_depth': 1, 'n_estimators': 50, 'subsample': 1}
Best score: 0.83869


In [ ]:
grid_search(params={'subsample':[0.5, 0.6, 0.7, 0.8, 0.9, 1],
                    'min_child_weight':[1, 2, 3, 4, 5],
                    'learning_rate':[0.1, 0.2, 0.3, 0.4, 0.5],
                    'max_depth':[1, 2, 3, 4, 5],
                    'n_estimators':[2]})

Best params: {'learning_rate': 0.5, 'max_depth': 2, 'min_child_weight': 4, 'n_estimators': 2, 'subsample': 0.9}
Best score: 0.81224


In [ ]:
grid_search(params={'subsample':[0.5, 0.6, 0.7, 0.8, 0.9, 1],
                    'min_child_weight':[1, 2, 3, 4, 5],
                    'learning_rate':[0.1, 0.2, 0.3, 0.4, 0.5],
                    'max_depth':[1, 2, 3, 4, 5, None],
                    'n_estimators':[2, 25, 50, 75, 100]}, random=True)

Best params: {'subsample': 0.6, 'n_estimators': 25, 'min_child_weight': 4, 'max_depth': 4, 'learning_rate': 0.5}
Best score: 0.82208


In [ ]:
grid_search(params={'colsample_bytree':[0.5, 0.6, 0.7, 0.8, 0.9, 1],
                    'max_depth':[1],
                    'n_estimators':[50]})

Best params: {'colsample_bytree': 1, 'max_depth': 1, 'n_estimators': 50}
Best score: 0.83869


In [ ]:
grid_search(params={'colsample_bylevel':[0.5, 0.6, 0.7, 0.8, 0.9, 1],
                    'max_depth':[1],
                    'n_estimators':[50]})

Best params: {'colsample_bylevel': 1, 'max_depth': 1, 'n_estimators': 50}
Best score: 0.83869


In [ ]:
grid_search(params={'colsample_bynode':[0.5, 0.6, 0.7, 0.8, 0.9, 1],
                    'colsample_bylevel':[0.5, 0.6, 0.7, 0.8, 0.9, 1],
                    'colsample_bytree':[0.5, 0.6, 0.7, 0.8, 0.9, 1],
                    'max_depth':[1],
                    'n_estimators':[50]})

Best params: {'colsample_bylevel': 0.9, 'colsample_bynode': 0.5, 'colsample_bytree': 0.8, 'max_depth': 1, 'n_estimators': 50}
Best score: 0.84852


In [ ]:
grid_search(params={'gamma':[0, 0.01, 0.05, 0.1, 0.5, 1, 2, 3],
                    'colsample_bylevel':[0.9],
                    'colsample_bytree':[0.8],
                    'colsample_bynode':[0.5],
                    'max_depth':[1],
                    'n_estimators':[50]})

Best params: {'colsample_bylevel': 0.9, 'colsample_bynode': 0.5, 'colsample_bytree': 0.8, 'gamma': 0, 'max_depth': 1, 'n_estimators': 50}
Best score: 0.84852
